# Transfer Learning with GNN (PyTorch)

This tutorial shows how to:
1. Pretrain a GNN model (NMPN) on the **ESOL** dataset.
2. Transfer learned weights to a new model.
3. Freeze the GNN backbone and fine-tune only the output MLP on the **FreeSolv** dataset.

The workflow follows the PyTorch pattern of using `model.load_state_dict()`,
`param.requires_grad = False`, and selective training of specific submodules.

In [ ]:
import importlib
import torch
import torch.nn as nn
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from torch_geometric.loader import DataLoader

import kgcnn_torch.training.trainer as _trainer_mod
importlib.reload(_trainer_mod)

from kgcnn_torch.models.nmpn import NMPNModel
from kgcnn_torch.training.trainer import fit
from kgcnn_torch.training.scheduler import LinearLearningRateScheduler, LinearWarmupLinearLearningRateScheduler
from kgcnn_torch.utils.devices import get_device
from kgcnn_torch.utils.plots import plot_train_test_loss

device = get_device("auto")
print(f"Using device: {device}")

## 1. Load the ESOL dataset for pretraining

In [ ]:
from kgcnn_torch.data.datasets.ESOLDataset import ESOLDataset

esol = ESOLDataset()
print(f"ESOL: {len(esol)} molecules")
print(f"Example graph: {esol[0]}")

## 2. Create and pretrain NMPN on ESOL

We use the NMPN (Neural Message Passing Network) architecture with Set2Set pooling.
The model expects PyG Data objects with:
- `data.z` or `data.x`: Node features (atomic numbers or one-hot)
- `data.edge_index`: Edge connectivity (2, M)
- `data.edge_attr`: Edge features (M, F)
- `data.batch`: Batch assignment (N,)
- `data.y`: Target labels

In [ ]:
# Train/test split (fixed seed for reproducibility)
indices = np.arange(len(esol))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

esol_train = esol[torch.tensor(train_idx).long()]
esol_test = esol[torch.tensor(test_idx).long()]

# IMPORTANT: Convert to list first! PyG InMemoryDataset.__getitem__ returns
# a NEW Data object each time, so in-place modification on dataset[i] is lost.
esol_train_list = [esol_train[i] for i in range(len(esol_train))]
esol_test_list = [esol_test[i] for i in range(len(esol_test))]

# Label scaling
y_train_np = np.array([d.y.numpy() for d in esol_train_list]).reshape(-1, 1)
scaler = StandardScaler()
scaler.fit(y_train_np)

print(f"Scaler: mean={scaler.mean_[0]:.4f}, std={scaler.scale_[0]:.4f}")

# Apply scaling to list elements (now modifications persist)
for data_list in [esol_train_list, esol_test_list]:
    for d in data_list:
        y_val = d.y.numpy().reshape(-1, 1)
        d.y = torch.tensor(scaler.transform(y_val), dtype=torch.float32).squeeze(0)

# Verify scaling worked
print(f"Before scaling (esol[0].y): {esol[0].y.item():.4f}")
print(f"After scaling (esol_train_list[0].y): {esol_train_list[0].y.item():.4f}")

train_loader = DataLoader(esol_train_list, batch_size=32, shuffle=True)
test_loader = DataLoader(esol_test_list, batch_size=32)

print(f"Train: {len(esol_train_list)}, Test: {len(esol_test_list)}")

In [ ]:
# Create the NMPN model
model_pre_trained = NMPNModel(
    node_dim=128,
    depth=3,
    units=128,                # Must match node_dim (Keras uses node_dim for both)
    edge_dim=11,              # 11 edge features from molecular encoding
    edge_mlp_units=[64, 64],
    edge_mlp_activation="silu",
    use_set2set=True,
    set2set_channels=64,
    set2set_T=3,
    output_units=[64, 32],
    output_activation="silu",
    output_final_activation="linear",
    num_targets=1,
    use_node_embedding=False,
    node_input_dim=41,        # 41-dim node features (matching Keras node_attributes)
)
model_pre_trained = model_pre_trained.to(device)
print(model_pre_trained)

In [ ]:
# Diagnostic: verify data shapes, model parameters, and initialization
sample = esol[0]
print("=== Data Diagnostics ===")
print(f"data.x shape: {sample.x.shape if sample.x is not None else 'None'}")
print(f"data.edge_attr shape: {sample.edge_attr.shape if sample.edge_attr is not None else 'None'}")
print(f"data.edge_index shape: {sample.edge_index.shape}")

# Check parameter count and shapes
total = sum(p.numel() for p in model_pre_trained.parameters())
print(f"\n=== Model Diagnostics ===")
print(f"Total parameters: {total:,}")
for name, p in model_pre_trained.named_parameters():
    print(f"  {name}: {p.shape} ({p.numel():,})")

# Verify Keras-compatible initialization
print("\n=== Initialization Check ===")
for name, module in model_pre_trained.named_modules():
    if isinstance(module, torch.nn.GRUCell):
        wh_std = module.weight_hh.data.std().item()
        bih_max = module.bias_ih.data.abs().max().item()
        print(f"  GRUCell ({name}): weight_hh std={wh_std:.4f} (orthogonal ~0.04-0.06), bias_ih max={bih_max:.6f} (should be 0)")
    elif isinstance(module, torch.nn.LSTMCell):
        wh_std = module.weight_hh.data.std().item()
        hs = module.hidden_size
        forget_bias = module.bias_ih.data[hs:2*hs].mean().item()
        print(f"  LSTMCell ({name}): weight_hh std={wh_std:.4f} (orthogonal), forget_bias mean={forget_bias:.1f} (should be 1.0)")

# Quick forward pass
model_pre_trained.eval()
batch = next(iter(train_loader)).to(device)
with torch.no_grad():
    out = model_pre_trained(batch)
print(f"\nOutput shape: {out.shape}")
print(f"Output sample: {out[:5].squeeze()}")

In [ ]:
# Define unscaled metrics (computed after inverse_transform by the trainer)
def _scaled_mae(pred, target):
    return torch.mean(torch.abs(pred - target))

def _scaled_rmse(pred, target):
    return torch.sqrt(torch.mean((pred - target) ** 2))

metrics = {
    "scaled_mean_absolute_error": _scaled_mae,
    "scaled_root_mean_squared_error": _scaled_rmse,
}

# Pretrain on ESOL
optimizer = torch.optim.Adam(model_pre_trained.parameters(), lr=1e-3)
scheduler = LinearLearningRateScheduler(
    optimizer, learning_rate_start=1e-3, learning_rate_stop=1e-5,
    epo_min=100, epo=300
)

history_pretrain = fit(
    model_pre_trained,
    train_loader=train_loader,
    val_loader=test_loader,
    optimizer=optimizer,
    loss_fn=nn.L1Loss(),
    scheduler=scheduler,
    epochs=300,
    device=device,
    verbose=1,
    metrics=metrics,
    scaler=scaler,
    compute_train_metrics=True,
)

# Add learning_rate alias for plotting
history_pretrain["learning_rate"] = history_pretrain["lr"]

# Plot all metrics (matching Keras plot style)
plot_train_test_loss(
    [history_pretrain],
    loss_name=["train_loss", "train_scaled_mean_absolute_error", "train_scaled_root_mean_squared_error", "learning_rate"],
    val_loss_name=["val_loss", "val_scaled_mean_absolute_error", "val_scaled_root_mean_squared_error"],
    dataset_name="ESOL",
    model_name="NMPN (pretraining)"
);

## 3. Save pretrained weights

In PyTorch we use `torch.save()` with `model.state_dict()` to persist the model weights.

In [ ]:
# Save pretrained weights
torch.save(model_pre_trained.state_dict(), "nmpn_esol_pretrained.pt")
print("Pretrained weights saved to nmpn_esol_pretrained.pt")

# Show the named parameters of the pretrained model
print("\nModel parameters:")
for name, param in model_pre_trained.named_parameters():
    print(f"  {name}: {param.shape}, trainable={param.requires_grad}")

## 4. Transfer learning: fine-tune on FreeSolv

Now we:
1. Create a new NMPN model with the **same architecture**.
2. Load pretrained weights with `model.load_state_dict()`.
3. Freeze the GNN backbone by setting `param.requires_grad = False`.
4. Keep only the output MLP trainable.
5. Train on the FreeSolv dataset.

In [ ]:
from kgcnn_torch.data.datasets.FreeSolvDataset import FreeSolvDataset

freesolv = FreeSolvDataset()
print(f"FreeSolv: {len(freesolv)} molecules")

In [ ]:
# Create a new model with the same architecture
model = NMPNModel(
    node_dim=128,
    depth=3,
    units=128,
    edge_dim=11,
    edge_mlp_units=[64, 64],
    edge_mlp_activation="silu",
    use_set2set=True,
    set2set_channels=64,
    set2set_T=3,
    output_units=[64, 32],
    output_activation="silu",
    output_final_activation="linear",
    num_targets=1,
    use_node_embedding=False,
    node_input_dim=41,
)

# Load pretrained weights
state_dict = torch.load("nmpn_esol_pretrained.pt", weights_only=True)
model.load_state_dict(state_dict)
print("Loaded pretrained weights.")

In [ ]:
# Step 1: Freeze ALL parameters
for param in model.parameters():
    param.requires_grad = False

# Step 2: Unfreeze only the output MLP
for param in model.output_mlp.parameters():
    param.requires_grad = True

# Verify which parameters are trainable
print("Trainable parameters after freezing backbone:")
total_params = 0
trainable_params = 0
for name, param in model.named_parameters():
    total_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()
        print(f"  [TRAINABLE] {name}: {param.shape}")

print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")

## 5. Train on FreeSolv with K-Fold Cross-Validation

We use a 5-fold cross-validation to evaluate the transfer learning approach.
For each fold, we:
1. Create a fresh model and load pretrained weights.
2. Freeze the backbone, keep only the output MLP trainable.
3. Train with a warmup learning rate schedule.

In [ ]:
kf = KFold(n_splits=5, random_state=42, shuffle=True)
history_list = []

# Reuse the same metric functions from pretraining
def _scaled_mae(pred, target):
    return torch.mean(torch.abs(pred - target))

def _scaled_rmse(pred, target):
    return torch.sqrt(torch.mean((pred - target) ** 2))

fold_metrics = {
    "scaled_mean_absolute_error": _scaled_mae,
    "scaled_root_mean_squared_error": _scaled_rmse,
}

for fold, (train_idx, test_idx) in enumerate(kf.split(np.arange(len(freesolv)))):
    print(f"\n{'='*60}")
    print(f"Fold {fold + 1}/5")
    print(f"{'='*60}")

    # Create fresh model with pretrained weights
    fold_model = NMPNModel(
        node_dim=128, depth=3, units=128, edge_dim=11,
        edge_mlp_units=[64, 64], edge_mlp_activation="silu",
        use_set2set=True, set2set_channels=64, set2set_T=3,
        output_units=[64, 32], output_activation="silu",
        output_final_activation="linear", num_targets=1,
        use_node_embedding=False, node_input_dim=41,
    )
    fold_model.load_state_dict(
        torch.load("nmpn_esol_pretrained.pt", weights_only=True)
    )

    # Freeze backbone, unfreeze output MLP
    for param in fold_model.parameters():
        param.requires_grad = False
    for param in fold_model.output_mlp.parameters():
        param.requires_grad = True

    fold_model = fold_model.to(device)

    # Prepare data - convert to list first to allow in-place label modification
    fold_train_sub = freesolv[torch.tensor(train_idx).long()]
    fold_test_sub = freesolv[torch.tensor(test_idx).long()]
    fold_train_list = [fold_train_sub[i] for i in range(len(fold_train_sub))]
    fold_test_list = [fold_test_sub[i] for i in range(len(fold_test_sub))]

    # Scale labels
    y_tr = np.array([d.y.numpy() for d in fold_train_list]).reshape(-1, 1)
    fold_scaler = StandardScaler()
    fold_scaler.fit(y_tr)
    for data_list in [fold_train_list, fold_test_list]:
        for d in data_list:
            y_val = d.y.numpy().reshape(-1, 1)
            d.y = torch.tensor(fold_scaler.transform(y_val), dtype=torch.float32).squeeze(0)

    fold_train_loader = DataLoader(fold_train_list, batch_size=32, shuffle=True)
    fold_test_loader = DataLoader(fold_test_list, batch_size=32)

    # Only optimize trainable parameters
    opt = torch.optim.Adam(
        filter(lambda p: p.requires_grad, fold_model.parameters()),
        lr=1e-3
    )
    sched = LinearWarmupLinearLearningRateScheduler(
        opt, learning_rate_start=1e-3, learning_rate_stop=1e-5,
        epo_warmup=5, epo=300
    )

    hist = fit(
        fold_model,
        train_loader=fold_train_loader,
        val_loader=fold_test_loader,
        optimizer=opt,
        loss_fn=nn.L1Loss(),
        scheduler=sched,
        epochs=300,
        device=device,
        verbose=1,
        metrics=fold_metrics,
        scaler=fold_scaler,
        compute_train_metrics=True,
    )
    # Add learning_rate alias
    hist["learning_rate"] = hist["lr"]
    history_list.append(hist)

plot_train_test_loss(
    history_list,
    loss_name=["train_loss", "train_scaled_mean_absolute_error", "train_scaled_root_mean_squared_error", "learning_rate"],
    val_loss_name=["val_loss", "val_scaled_mean_absolute_error", "val_scaled_root_mean_squared_error"],
    dataset_name="FreeSolv",
    model_name="NMPN (transfer learning)"
);

## 6. Verify which weights changed

We compare the weights of the fine-tuned model against the pretrained weights.
Only the output MLP parameters should have changed; all backbone weights should
remain identical.

In [ ]:
# Load original pretrained weights for comparison
original_state = torch.load("nmpn_esol_pretrained.pt", weights_only=True)
finetuned_state = fold_model.state_dict()

print("Weight comparison (max absolute difference per parameter):")
print("-" * 70)
for name in original_state:
    diff = torch.max(torch.abs(
        original_state[name].cpu().float() - finetuned_state[name].cpu().float()
    )).item()
    status = "CHANGED" if diff > 0 else "FROZEN"
    print(f"  [{status:7s}] {name}: max_diff={diff:.6f}")

## Summary

Transfer learning pattern in PyTorch with kgcnn-torch:

```python
# 1. Save pretrained weights
torch.save(model.state_dict(), "pretrained.pt")

# 2. Load into new model
new_model = NMPNModel(...)  # same architecture
new_model.load_state_dict(torch.load("pretrained.pt", weights_only=True))

# 3. Freeze backbone
for param in new_model.parameters():
    param.requires_grad = False

# 4. Unfreeze output MLP
for param in new_model.output_mlp.parameters():
    param.requires_grad = True

# 5. Only pass trainable params to optimizer
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, new_model.parameters()), lr=1e-3
)
```